In [0]:
%sql
SELECT
    status_cartao,
    status_conta,
    COUNT(*) AS qtd_transacoes,
    SUM(valor) AS valor_total
FROM novarota.ouro.gold_transacoes_cartao
GROUP BY
    status_cartao,
    status_conta
ORDER BY
    qtd_transacoes DESC;

In [0]:
%sql
SELECT
    t.id_transacao,
    t.data_transacao,
    t.valor,
    c.id_cartao,
    c.status_cartao,
    c.data_emissao,
    c.data_atualizacao
FROM novarota.prata.transacoes t
INNER JOIN novarota.prata.cartoes c
    ON t.id_cartao = c.id_cartao
WHERE c.status_cartao IN ('CANCELADO', 'BLOQUEADO')
  AND t.estornada = FALSE
ORDER BY t.data_transacao;

In [0]:
%sql
SELECT
    e.id_estorno,
    e.id_transacao,
    e.data_estorno,
    t.data_transacao,
    t.valor
FROM novarota.prata.estornos e
INNER JOIN novarota.prata.transacoes t
    ON e.id_transacao = t.id_transacao
WHERE e.data_estorno < CAST(t.data_transacao AS DATE)
ORDER BY e.data_estorno;

In [0]:
%sql
SELECT
    COUNT(*) AS estornos_incorretamente_aceitos
FROM novarota.prata.estornos e
INNER JOIN novarota.prata.quarentena q
    ON e.id_estorno = q.chave_registro
WHERE q.entidade = 'estornos'
  AND q.motivo_erro LIKE '%ESTORNO_ANTERIOR_TRANSACAO%';

In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM novarota.prata.transacoes
     WHERE estornada = FALSE) AS qtd_prata_validas,

    (SELECT COUNT(*)
     FROM novarota.ouro.gold_transacoes_cartao) AS qtd_ouro,

    (SELECT SUM(valor)
     FROM novarota.prata.transacoes
     WHERE estornada = FALSE) AS valor_prata,

    (SELECT SUM(valor)
     FROM novarota.ouro.gold_transacoes_cartao) AS valor_ouro;